# Sionna 0.19 — 915.95 MHz Simulation (Nottingham)

**Frequency:** 915.95 MHz · **RX:** 1200 sequential from CSV · **Terrain:** flat · **Materials:** metal roof, brick walls, wet ground

**CSV:** nottingham915.csv  ·  TX EIRP=55.1 dBm  ·  RX system gain=−7.8 dB

## Cell 0 — Imports

In [ ]:
# GPU memory growth — must be set before any TF/CUDA import
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [ ]:
import os, sys, json, csv, time, warnings, glob, re
import numpy as np
import pandas as pd
import matplotlib; matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 100})
import matplotlib.pyplot as plt
from scipy.constants import speed_of_light as C
from pyproj import Transformer
import math
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

FORCE_CPU_RT = False
if not FORCE_CPU_RT:
    import mitsuba as mi
    mi.set_variant('cuda_ad_rgb')
else:
    import mitsuba as mi
    mi.set_variant('llvm_ad_rgb')
import drjit as dr

import sionna
import tensorflow as tf
from sionna.rt import load_scene, RadioMaterial, PlanarArray, Transmitter, Receiver
from sionna.rt.antenna import iso_pattern, dipole_pattern
print(f'Sionna {sionna.__version__}  TF {tf.__version__}  Mitsuba {mi.__version__}')

## Cell 1 — Configuration

In [ ]:
# ── City / scene ─────────────────────────────────────────────────────────────
CITY_NAME    = 'Nottingham'
UTM_EPSG     = 32630          # UTM zone 30N (UK)

# ── Scene bbox (WGS84) ────────────────────────────────────────────────────────
# Tight bbox: TX + first 1200 RX + 900m margin (10.1×7.3 km = 74 km²)
SCENE_WEST   = -1.260093
SCENE_EAST   = -1.129307
SCENE_SOUTH  =  52.945798
SCENE_NORTH  =  52.998702

# ── TX parameters (from nottingham915.csv header) ─────────────────────────────
# Amp power=50.3 dBm, cable loss=1.3 dB → conducted=49.0 dBm
# Antenna gain=1.3 dBi → EIRP=50.3 dBm (per CSV header)
TX_LON              = -1.2559
TX_LAT              =  52.9863
TX_AGL_M            = 17.0           # m  (CSV: Tx antenna height = 17 m)
TX_CONDUCTED_DBM    = 49.0           # dBm (50.3 amp − 1.3 cable loss)
TX_ANTENNA_GAIN_DBI =  1.3           # dBi (collinear omni, per CSV)
EIRP_DBM            = TX_CONDUCTED_DBM + TX_ANTENNA_GAIN_DBI  # 50.3 dBm

# ── Antenna pattern ───────────────────────────────────────────────────────────
# 'donut' → half-wave dipole shape scaled to TX_ANTENNA_GAIN_DBI
#           max at horizon, deep null at zenith/nadir — matches collinear mast
# 'iso'   → isotropic 0 dBi (uniform sphere) — safe fallback
ANTENNA_PATTERN = 'donut'

# ── RX parameters (from nottingham915.csv header) ─────────────────────────────
# RX antenna gain=-1 dBi, cable loss=0.2 dB, splitter=6.1 dB,
# LNA=0 dB, BPF=0.5 dB → system gain = -1 - 0.2 - 6.1 + 0 - 0.5 = -7.8 dB
RX_AGL_M           =  1.5            # m  (CSV: Rx antenna height = 1.5 m)
RX_EXTRA_GAIN_DB   = -7.8            # dB (system gain: antenna + cable + filters)
SITE_CORRECTION_DB =  0.0            # dB (calibration offset — adjust after bias check)
NOISE_FLOOR_DBM    = -124.0          # dBm (CSV: System noise floor)

# ── RX selection ──────────────────────────────────────────────────────────────
NUM_RX       = 1200              # first 1200 sequential rows from CSV

# ── Frequency ─────────────────────────────────────────────────────────────────
FREQUENCY_HZ = 915.95e6          # Hz  (CSV: Frequency = 915.95 MHz)

# ── Terrain ───────────────────────────────────────────────────────────────────
FLAT_TERRAIN = True              # flat z=0 plane

# ── Ray tracing ───────────────────────────────────────────────────────────────
MAX_DEPTH        = 6             # bounces
NUM_SAMPLES_PS   = 10_000_000    # rays per batch
SCAT_KEEP_PROB   = 0.001         # scatter fraction (energy corrected)
BATCH_SIZE       = 5             # RX per compute_paths() call

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = '/home/georgeskai/Documents/FYP2026/nottingham900'
SCENE_DIR       = os.path.join(BASE_DIR, 'scene')
SCENE_XML       = os.path.join(SCENE_DIR, 'scene.xml')
OUT_DIR         = os.path.join(BASE_DIR, 'results')
os.makedirs(OUT_DIR, exist_ok=True)

# Source measurement CSV (nottingham915.csv)
OFCOM_RAW_CSV   = '/home/georgeskai/Documents/FYP2026/nottingham900/nottingham915.csv'

# Output files (same naming convention as main notebook)
RX_CSV          = os.path.join(SCENE_DIR, 'receiver_locations.csv')
MEASUREMENT_CSV = os.path.join(SCENE_DIR, 'measurements_with_pathloss.csv')

print(f'Frequency        : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX               : lon={TX_LON}  lat={TX_LAT}  AGL={TX_AGL_M}m')
print(f'TX conducted     : {TX_CONDUCTED_DBM} dBm   antenna={TX_ANTENNA_GAIN_DBI} dBi   EIRP={EIRP_DBM:.1f} dBm')
print(f'Antenna pattern  : {ANTENNA_PATTERN}')
print(f'RX system gain   : {RX_EXTRA_GAIN_DB} dB')
print(f'Noise floor      : {NOISE_FLOOR_DBM} dBm')
print(f'NUM_RX           : {NUM_RX}')
print(f'BASE_DIR         : {BASE_DIR}')
print(f'SCENE_XML        : {SCENE_XML}')
print(f'OFCOM_RAW_CSV    : {OFCOM_RAW_CSV}')

## Cell 2 — Coordinate Utilities (GPS → UTM → Local, NO BNG)

In [ ]:
# ── Coordinate transformers (pyproj, WGS84 ↔ UTM 30N) ──────────────────────
# Best practice: always_xy=True enforces (lon, lat) / (easting, northing) order
# regardless of the CRS axis convention — prevents silent axis swaps.
gps_to_utm = Transformer.from_crs('EPSG:4326', f'EPSG:{UTM_EPSG}', always_xy=True)
utm_to_gps = Transformer.from_crs(f'EPSG:{UTM_EPSG}', 'EPSG:4326', always_xy=True)

# Scene centre in UTM (origin of local coordinate system)
center_lon = (SCENE_WEST + SCENE_EAST)  / 2
center_lat = (SCENE_SOUTH + SCENE_NORTH) / 2
utm_center_x, utm_center_y = gps_to_utm.transform(center_lon, center_lat)

def gps_to_local(lon, lat, height=0.0):
    """WGS84 (lon, lat) → scene-local (x, y, z) in metres.
    Origin = scene bbox centre. X = east, Y = north, Z = up.
    """
    ux, uy = gps_to_utm.transform(lon, lat)
    return float(ux - utm_center_x), float(uy - utm_center_y), float(height)

def local_to_gps(x, y):
    """Scene-local (x, y) → WGS84 (lon, lat)."""
    lon, lat = utm_to_gps.transform(x + utm_center_x, y + utm_center_y)
    return float(lon), float(lat)

print(f'Scene centre  : lon={center_lon:.5f}  lat={center_lat:.5f}')
print(f'UTM centre    : ({utm_center_x:.1f}, {utm_center_y:.1f})')
print(f'Test gps_to_local(TX): {gps_to_local(TX_LON, TX_LAT)[:2]}')

## Cell 3 — Load Scene

In [ ]:
# ── Clear GPU memory before loading ──────────────────────────────────────────
import gc
tf.keras.backend.clear_session()
gc.collect()
try:
    import drjit as _dr
    _dr.sync_thread()
except Exception:
    pass

# ── Scene size check ──────────────────────────────────────────────────────────
# The nottingham3602 scene (16×10 km) is too large for GPUs with <16 GB.
# Use the scene builder output (nottingham900, ~10×7 km) instead.
# If SCENE_XML points at the wrong scene, update BASE_DIR / SCENE_XML in Cell 1.
if not os.path.exists(SCENE_XML):
    raise FileNotFoundError(
        f'Scene not found: {SCENE_XML}\n'
        'Run sionna019_scene_builder.ipynb (Cells 0–6) to generate the scene first.\n'
        'Do NOT point SCENE_XML at nottingham3602 — it is too large for GPU memory.')

_mesh_dir = os.path.join(os.path.dirname(SCENE_XML), 'meshes')
_scene_mb = sum(
    os.path.getsize(os.path.join(_mesh_dir, f))
    for f in os.listdir(_mesh_dir) if f.endswith('.ply')
) / 1024 / 1024 if os.path.isdir(_mesh_dir) else 0
print(f'Scene PLY total : {_scene_mb:.1f} MB  ({SCENE_XML})')
if _scene_mb > 400:
    print('WARNING: scene >400 MB — if GPU OOM occurs set FORCE_CPU_RT=True in Cell 0')

# ── Load scene ────────────────────────────────────────────────────────────────
print('Loading scene ...')
scene = load_scene(SCENE_XML)
scene.frequency       = FREQUENCY_HZ
scene.synthetic_array = False

# ── Customised antenna patterns (Ofcom 915 MHz drive-test) ───────────────────
# TX : collinear omni mast  — donut, scaled to TX_ANTENNA_GAIN_DBI (+1.3 dBi)
#      F_theta = scale * cos(pi/2 * cos(theta)) / sin(theta),  F_phi = 0
#      Max at horizon (theta=90°), deep null at zenith/nadir
# RX : vehicle rooftop omni — isotropic (0 dBi)
#      Equipment losses (cable, splitter, BPF) folded into RX_EXTRA_GAIN_DB
_D_HW = 1.6409   # half-wave dipole directivity (linear) = 2.15 dBi

def _tx_pattern_915(theta, phi):
    """Ofcom 915 MHz TX — donut omni scaled to TX_ANTENNA_GAIN_DBI."""
    _scale = np.float32((10 ** (TX_ANTENNA_GAIN_DBI / 10) / _D_HW) ** 0.5)
    cos_t  = tf.cos(theta)
    sin_t  = tf.sin(theta)
    safe_s = tf.where(tf.abs(sin_t) < 1e-6,
                      tf.ones_like(sin_t) * 1e-6, sin_t)
    f_theta = tf.cast(_scale * tf.cos(np.float32(np.pi / 2) * cos_t) / safe_s,
                      tf.complex64)
    f_phi   = tf.zeros_like(f_theta)
    return f_theta, f_phi

def _rx_pattern_915(theta, phi):
    """Ofcom 915 MHz RX — isotropic (losses in RX_EXTRA_GAIN_DB)."""
    _iso  = tf.cast(tf.ones_like(theta) / np.float32(np.sqrt(4.0 * np.pi)),
                    tf.complex64)
    f_phi = tf.zeros_like(_iso)
    return _iso, f_phi

def _make_antenna_array(pattern_fn, fallback='hw_dipole'):
    """Build 1×1 PlanarArray; falls back to named built-in if callable rejected."""
    try:
        arr = PlanarArray(num_rows=1, num_cols=1,
                          vertical_spacing=0.5, horizontal_spacing=0.5,
                          pattern=pattern_fn, polarization='V')
        return arr, 'callable'
    except Exception as _e:
        arr = PlanarArray(num_rows=1, num_cols=1,
                          vertical_spacing=0.5, horizontal_spacing=0.5,
                          pattern=fallback, polarization='V')
        return arr, f'fallback={fallback} ({_e})'

_ant_cfg = globals().get('ANTENNA_PATTERN', 'donut')
if _ant_cfg == 'donut':
    scene.tx_array, _tx_mode = _make_antenna_array(_tx_pattern_915, fallback='hw_dipole')
    scene.rx_array, _rx_mode = _make_antenna_array(_rx_pattern_915, fallback='iso')
else:
    scene.tx_array, _tx_mode = _make_antenna_array(iso_pattern, fallback='iso')
    scene.rx_array, _rx_mode = _make_antenna_array(iso_pattern, fallback='iso')

print(f'Scene loaded   : {len(scene.objects)} objects')
print(f'Frequency      : {FREQUENCY_HZ/1e6:.2f} MHz')
print(f'TX antenna     : {_ant_cfg}  {TX_ANTENNA_GAIN_DBI:+.1f} dBi  [{_tx_mode}]')
print(f'RX antenna     : isotropic  0.0 dBi  system_gain={RX_EXTRA_GAIN_DB:.1f} dB  [{_rx_mode}]')
print(f'TX EIRP        : {EIRP_DBM:.1f} dBm')

## Cell 4 — Materials (Metal Roof, Brick Walls, Wet Ground)

In [ ]:
# ── ITU-R P.2040-2 at 900 MHz ────────────────────────────────────────────────
# eps_r and sigma computed at FREQUENCY_HZ
_f_ghz = FREQUENCY_HZ / 1e9   # 0.9 GHz

def _itu(a_eps, b_eps, c_sig, d_sig):
    eps   = a_eps * (_f_ghz ** b_eps)
    sigma = c_sig * (_f_ghz ** d_sig)
    return float(eps), float(sigma)

_mats = {
    # name              a_eps  b_eps   c_sig   d_sig     S_scat  note
    'itu_brick'      : (*_itu(3.75, 0, 0.038,  0    ), 0.20),   # walls
    'itu_metal'      : (*_itu(1.0,  0, 1e7,    0    ), 0.05),   # roofs (specular)
    'itu_wet_ground' : (*_itu(30.0, 0, 0.15,   0    ), 0.10),   # floor/terrain
    'itu_concrete'   : (*_itu(5.31, 0, 0.0326, 0.8095), 0.15), # generic
    'itu_glass'      : (*_itu(6.27, 0, 0.0043, 1.1925), 0.05), # windows
    'itu_vegetation' : (*_itu(1.7,  0, 0.050,  0.60 ), 0.50),   # trees
    'itu_asphalt'    : (*_itu(3.18, 0, 0.058,  0    ), 0.10),   # roads
    'itu_wood'       : (*_itu(1.99, 0, 0.0,    0    ), 0.15),   # timber
}

from sionna.rt.scattering_pattern import LambertianPattern as _LP
_lambertian = _LP()

for _name, (_eps, _sig, _s) in _mats.items():
    # scene.xml from the scene builder may have already registered these
    # materials — update existing ones instead of re-adding to avoid ValueError
    _existing = scene.radio_materials.get(_name)
    if _existing is not None:
        _existing.relative_permittivity  = _eps
        _existing.conductivity           = _sig
        _existing.scattering_coefficient = _s
        _existing.xpd_coefficient        = 0.0
        _existing.scattering_pattern     = _lambertian
        _tag = 'updated'
    else:
        _mat = RadioMaterial(_name,
                             relative_permittivity=_eps,
                             conductivity=_sig,
                             scattering_coefficient=_s,
                             xpd_coefficient=0.0,
                             scattering_pattern=_lambertian)
        scene.add(_mat)
        _tag = 'added'
    print(f'  {_name:<22} eps={_eps:.2f}  sigma={_sig:.4f}  S={_s:.2f}  [{_tag}]')

# Assign materials to scene objects by name pattern
# Roof → metal, Wall → brick, Ground/terrain → wet_ground
_assigned = {'roof': 0, 'wall': 0, 'terrain': 0, 'other': 0}
for _obj_name, _obj in scene.objects.items():
    _n = _obj_name.lower()
    if 'roof' in _n:
        _obj.radio_material = scene.radio_materials['itu_metal']
        _assigned['roof'] += 1
    elif 'wall' in _n or 'bld' in _n or 'building' in _n:
        _obj.radio_material = scene.radio_materials['itu_brick']
        _assigned['wall'] += 1
    elif 'terrain' in _n or 'ground' in _n or 'floor' in _n:
        _obj.radio_material = scene.radio_materials['itu_wet_ground']
        _assigned['terrain'] += 1
    else:
        _assigned['other'] += 1

print(f'\nMaterial assignment: roof={_assigned["roof"]}  wall={_assigned["wall"]}  '
      f'terrain={_assigned["terrain"]}  other={_assigned["other"]}')

## Cell 5 — Place TX (Flat Terrain, z=0)

In [ ]:
# Remove previous TX
for _n in list(scene.transmitters.keys()):
    scene.remove(_n)

tx_x, tx_y, _ = gps_to_local(TX_LON, TX_LAT)
tx_z = TX_AGL_M   # flat terrain: ground = 0

tx = Transmitter(name='tx0',
                 position=[tx_x, tx_y, tx_z],
                 orientation=[0.0, 0.0, 0.0])
scene.add(tx)

print(f'TX placed:')
print(f'  GPS      : lon={TX_LON}  lat={TX_LAT}')
print(f'  Local    : ({tx_x:.1f}, {tx_y:.1f}, {tx_z:.1f}) m')
print(f'  AGL      : {TX_AGL_M} m  (flat terrain)')

## Cell 6 — RX Extraction (1200 Nearest, Sequential)

In [ ]:
import csv as _csv_mod, math

print('=' * 60)
print('CELL 6 — RX EXTRACTION (first 1200 sequential from CSV)')
print('=' * 60)

if not os.path.exists(OFCOM_RAW_CSV):
    raise FileNotFoundError(f'CSV not found: {OFCOM_RAW_CSV}')

# ── Auto-detect header row ────────────────────────────────────────────────────
# Scan up to the first 40 lines for the row that contains both
# 'Latitude' and 'Longitude' — works regardless of how many metadata
# lines precede the data (the Ofcom format varies between campaigns).
_hdr_idx = None
with open(OFCOM_RAW_CSV, 'r', encoding='utf-8', errors='replace') as _f:
    for _i, _line in enumerate(_f):
        if 'Latitude' in _line and 'Longitude' in _line:
            _hdr_idx = _i
            break
        if _i > 40:
            break

if _hdr_idx is None:
    raise ValueError(
        f'Could not find header row in {OFCOM_RAW_CSV}\n'
        'Expected a line containing both "Latitude" and "Longitude" in the first 40 lines.')

print(f'Header at line {_hdr_idx + 1}  (0-based index {_hdr_idx})')
_df_raw = pd.read_csv(OFCOM_RAW_CSV, skiprows=_hdr_idx, low_memory=False)
print(f'Columns: {list(_df_raw.columns)}')
print(f'Total rows: {len(_df_raw)}')

# ── Column mapping — flexible: match by keyword ───────────────────────────────
def _find_col(df, *keywords):
    """Return first column name that contains all keywords (case-insensitive)."""
    for col in df.columns:
        c = col.strip().lower()
        if all(k.lower() in c for k in keywords):
            return col
    return None

_lat_col  = _find_col(_df_raw, 'latitude')
_lon_col  = _find_col(_df_raw, 'longitude')
_rssi_col = _find_col(_df_raw, 'measurement') or _find_col(_df_raw, 'dBm') or _find_col(_df_raw, 'dbm')

if not _lat_col:
    raise KeyError(f'No Latitude column found. Available: {list(_df_raw.columns)}')
if not _lon_col:
    raise KeyError(f'No Longitude column found. Available: {list(_df_raw.columns)}')
if not _rssi_col:
    raise KeyError(f'No RSSI/measurement column found. Available: {list(_df_raw.columns)}')

print(f'Lat  col : {_lat_col!r}')
print(f'Lon  col : {_lon_col!r}')
print(f'RSSI col : {_rssi_col!r}')

# Drop rows with non-numeric values in key columns
for _c in [_lat_col, _lon_col, _rssi_col]:
    _df_raw[_c] = pd.to_numeric(_df_raw[_c], errors='coerce')
_df_raw = _df_raw.dropna(subset=[_lat_col, _lon_col, _rssi_col]).reset_index(drop=True)
print(f'Rows after numeric filter: {len(_df_raw)}')

# Take first NUM_RX rows in CSV order (sequential as recorded)
_sel = _df_raw.head(NUM_RX).copy()

# Distance from TX (for info only)
_dlon_m = 111000.0 * math.cos(math.radians(TX_LAT))
_dlat_m = 111000.0
_sel['_dist_km'] = (
    ((_sel[_lat_col] - TX_LAT) * _dlat_m)**2 +
    ((_sel[_lon_col] - TX_LON) * _dlon_m)**2
)**0.5 / 1000.0

print(f'\nSelected  : {len(_sel)} receivers (rows 1–{len(_sel)}, sequential CSV order)')
print(f'Dist range: {_sel["_dist_km"].min():.3f} – {_sel["_dist_km"].max():.3f} km')
print(f'RSSI range: {_sel[_rssi_col].min():.1f} – {_sel[_rssi_col].max():.1f} dBm')

# ── Write receiver_locations.csv ──────────────────────────────────────────────
os.makedirs(os.path.dirname(RX_CSV), exist_ok=True)
with open(RX_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'height'])
    for _idx, _row in _sel.iterrows():
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     RX_AGL_M])
print(f'\nWritten : {RX_CSV}  ({len(_sel)} rows)')

# ── Write measurements_with_pathloss.csv ──────────────────────────────────────
with open(MEASUREMENT_CSV, 'w', newline='') as _f:
    _w = _csv_mod.writer(_f)
    _w.writerow(['name', 'lon', 'lat', 'local_measurement_dBm', 'path_loss_dB'])
    for _idx, _row in _sel.iterrows():
        _rssi = float(_row[_rssi_col])
        _pl   = EIRP_DBM - _rssi + RX_EXTRA_GAIN_DB
        _w.writerow([f'RX_{_idx:06d}',
                     f'{float(_row[_lon_col]):.6f}',
                     f'{float(_row[_lat_col]):.6f}',
                     f'{_rssi:.2f}',
                     f'{_pl:.2f}'])
print(f'Written : {MEASUREMENT_CSV}  ({len(_sel)} rows)')

## Cell 7 — Place Receivers

In [ ]:
import time as _time

df_rx = pd.read_csv(RX_CSV)
for nm in list(scene.receivers.keys()):
    scene.remove(nm)

receivers = []
for _, row in df_rx.iterrows():
    lx, ly, _ = gps_to_local(float(row['lon']), float(row['lat']))
    lz = RX_AGL_M   # flat terrain
    rx = Receiver(name=row['name'], position=[lx, ly, lz])
    scene.add(rx)
    receivers.append(rx)

print(f'Placed {len(receivers)} receivers at z={RX_AGL_M}m (flat terrain)')

## Cell DIAG — Step-by-Step Bias Diagnostic

Run **before the path solver** to verify TX/RX positions, antenna heights, and scene geometry.
Tests 10 → 100 receivers to catch systematic bias early.

In [ ]:
# ====================================================================
# CELL DIAG — Step-by-Step Bias Diagnostic (10 → 100 receivers)
# ====================================================================
# Tests RSSI formula, RX heights, TX position, and geometry systematically.
# Run BEFORE CELL 9b to isolate the source of high RMSE.
# ====================================================================
import numpy as np, pandas as pd, math, os, time
from pyproj import Transformer as _Tr

print("=" * 70)
print("BIAS DIAGNOSTIC — Step-by-step RMSE decomposition")
print("=" * 70)

# ── Load measurements ─────────────────────────────────────────────────────────
_df_meas = pd.read_csv(MEASUREMENT_CSV)
print(f"\nMeasurements loaded: {len(_df_meas)} rows")
print(f"  RSSI range   : {_df_meas['local_measurement_dBm'].min():.1f} → {_df_meas['local_measurement_dBm'].max():.1f} dBm")
print(f"  PL range     : {_df_meas['path_loss_dB'].min():.1f} → {_df_meas['path_loss_dB'].max():.1f} dB")

# ── STEP 1: Formula check — FSPL vs measured at known distances ────────────────
print("\n" + "─" * 60)
print("STEP 1 — Free-Space Path Loss formula validation")
print("─" * 60)
C = 3e8
_f = FREQUENCY_HZ
_fspl = lambda d: 20*np.log10(4*np.pi*d*_f/C)

_tx_lon, _tx_lat = TX_LON, TX_LAT
_tx_x, _tx_y = gps_to_utm.transform(_tx_lon, _tx_lat)

# Compute distance for each measurement
_rx_x = np.array([gps_to_utm.transform(r['lon'], r['lat'])[0] for _, r in _df_meas.iterrows()])
_rx_y = np.array([gps_to_utm.transform(r['lon'], r['lat'])[1] for _, r in _df_meas.iterrows()])
_dist = np.sqrt((_rx_x - _tx_x)**2 + (_rx_y - _tx_y)**2)
_df_meas = _df_meas.copy()
_df_meas['dist_m'] = _dist

# Near receivers (50-300m) — most likely LOS → compare against FSPL
_near = _df_meas[_df_meas['dist_m'].between(50, 300)].copy()
_near['fspl_db'] = _near['dist_m'].apply(_fspl)
_near['measured_pl'] = _near['path_loss_dB']
_near['vs_fspl'] = _near['measured_pl'] - _near['fspl_db']
print(f"  Near receivers (50-300m): {len(_near)}")
print(f"  TX_CONDUCTED={TX_CONDUCTED_DBM:.1f} dBm  RX_EXTRA={RX_EXTRA_GAIN_DB:.1f} dB  SITE_CORR={SITE_CORRECTION_DB:.1f} dB")
print(f"  Sionna formula: RSSI = TX_CONDUCTED - PL + RX_EXTRA + SITE_CORR")
print(f"  (paths.a already includes TX/RX antenna gains — do NOT use EIRP or SYS_GAIN)")
print()
print(f"  {'Name':<12} {'Dist(m)':>8} {'RSSI(dBm)':>10} {'PL_meas(dB)':>12} {'FSPL(dB)':>9} {'PL-FSPL(dB)':>12}")
for _, r in _near.head(10).iterrows():
    print(f"  {str(r['name']):<12} {r['dist_m']:>8.0f} {r['local_measurement_dBm']:>10.1f} "
          f"{r['measured_pl']:>12.1f} {r['fspl_db']:>9.1f} {r['vs_fspl']:>12.1f}")
print(f"\n  Mean PL-FSPL (near): {_near['vs_fspl'].mean():+.1f} dB  "
      f"(expected +5 to +15 dB for urban LOS overhead)")
print(f"  If PL-FSPL < 0 → EIRP/SYS_GAIN overestimated")
print(f"  If PL-FSPL >> 20 dB → EIRP underestimated or RX underground")

# ── STEP 2: RX height check ────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 2 — Receiver height sanity check")
print("─" * 60)
_rxlist = list(scene.receivers.values())
if not _rxlist and os.path.exists(RX_CSV):
    _df_rx2 = pd.read_csv(RX_CSV)
    for _, _row2 in _df_rx2.iterrows():
        # Use pre-computed x_m, y_m, z_m from Cell 6c (terrain-corrected)
        _x2  = float(_row2['x_m'])
        _y2  = float(_row2['y_m'])
        _lz2 = float(_row2['z_m'])   # already terrain-corrected by Cell 6c
        _rx2 = Receiver(name=str(_row2['name']),
                        position=[_x2, _y2, _lz2])
        scene.add(_rx2)
    _rxlist = list(scene.receivers.values())
    print(f"  Loaded {len(_rxlist)} receivers from {RX_CSV} (terrain-corrected z)")
_heights = [_safe(rx.position[2]) for rx in _rxlist[:20]]
print(f"  First 20 RX heights (local Z, m):")
for rx, h in zip(_rxlist[:20], _heights):
    flag = " ⚠ UNDERGROUND" if h < -5 else (" ⚠ TOO HIGH (check terrain)" if h > 300 else "")
    print(f"    {rx.name:<12}  z={h:+.2f}m{flag}")
_neg = sum(1 for h in [_safe(rx.position[2]) for rx in _rxlist] if h < 0)
print(f"\n  Total RX underground (z<0): {_neg} / {len(_rxlist)}")

# ── STEP 3: TX position check ─────────────────────────────────────────────────
print("\n" + "─" * 60)
print("STEP 3 — TX position check")
print("─" * 60)
# Safe fallback for TX height variable name across notebook versions
if 'TX_AGL_M' not in dir(): TX_AGL_M = 17.0
if 'TX_LAT' not in dir(): TX_LAT = 52.9863
if 'TX_LON' not in dir(): TX_LON = -1.2559
_tx = list(scene.transmitters.values())[0]
_tx_lx, _tx_ly, _tx_lz = _safe(_tx.position[0]), _safe(_tx.position[1]), _safe(_tx.position[2])
_tx_glon, _tx_glat = local_to_gps(_tx_lx, _tx_ly)
print(f"  TX local  : ({_tx_lx:.1f}, {_tx_ly:.1f}, {_tx_lz:.1f}) m")
print(f"  TX GPS    : lat={_tx_glat:.6f}  lon={_tx_glon:.6f}")
print(f"  Expected  : lat={TX_LAT:.6f}  lon={TX_LON:.6f}  h={TX_AGL_M:.1f}m")
_lat_err = abs(_tx_glat - TX_LAT) * 111000
_lon_err = abs(_tx_glon - TX_LON) * 111000 * math.cos(math.radians(TX_LAT))
# TX z includes terrain offset — compare AGL only
_tx_terrain_est = getattr(tx, '_ground_z', _tx_lz - TX_AGL_M)
_tx_agl_actual  = _tx_lz - (_tx_lz - TX_AGL_M)  # = TX_AGL_M always
print(f"  Position error: {_lat_err:.1f}m N-S  {_lon_err:.1f}m E-W  Z={_tx_lz:.1f}m (terrain+AGL)")
print(f"  Terrain at TX  : {_tx_lz - TX_AGL_M:.1f}m  AGL={TX_AGL_M:.1f}m  Total={_tx_lz:.1f}m ✓")

# ── TX surroundings geometry check ───────────────────────────────────────────
print("\n  TX surroundings — ray cast in 8 directions (50m radius):")
_angles = [0, 45, 90, 135, 180, 225, 270, 315]
_dirs = ['N','NE','E','SE','S','SW','W','NW']
for _ang, _dname in zip(_angles, _dirs):
    import math as _m
    _dx = 50 * _m.cos(_m.radians(_ang))
    _dy = 50 * _m.sin(_m.radians(_ang))
    _probe_x = _tx_lx + _dx
    _probe_y = _tx_ly + _dy
    _gz_probe = ray_cast_ground_z(_probe_x, _probe_y)
    _bldg_h = _gz_probe - (_tx_lz - TX_AGL_M)  # height above TX terrain
    _visible = "LOS clear ✓" if _gz_probe < _tx_lz else f"BLOCKED (bldg top {_gz_probe:.1f}m > TX {_tx_lz:.1f}m)"
    print(f"    {_dname:>2}  ground/roof z={_gz_probe:6.1f}m  relative={_bldg_h:+5.1f}m  {_visible}")
_tx_h_err = 0.0  # height is always correct with terrain correction
print(f"  ✓ TX height OK (terrain-corrected)")
if _lat_err > 50 or _lon_err > 50:
    print("  ⚠ TX position error > 50m — check GPS→UTM→local conversion")
else:
    print("  ✓ TX position OK")

# ── STEP 4: Quick path solver on 50 receivers across bands ─────────────────────────
print("\n" + "─" * 60)
print("STEP 4 — Path solver: 50 receivers across distance bands")
print("─" * 60)
# Sample 50 receivers spread across distance bands for geometry validation
# Full physics: los + reflection + diffraction + edge_diffraction + scattering
_df_meas2 = _df_meas.copy()
_xy = _df_meas2.apply(lambda r: gps_to_utm.transform(float(r['lon']), float(r['lat'])), axis=1)
_df_meas2['_lx'] = [xy[0] - utm_center_x for xy in _xy]
_df_meas2['_ly'] = [xy[1] - utm_center_y for xy in _xy]
_df_meas2['_dtx'] = np.sqrt((_df_meas2['_lx'] - _tx_lx)**2 + (_df_meas2['_ly'] - _tx_ly)**2)

# Sample across distance bands: 10 near, 10 mid-near, 10 mid, 10 mid-far, 10 far
_df_meas2 = _df_meas2.sort_values('_dtx').reset_index(drop=True)
_n_total  = len(_df_meas2)
_bands = [
    _df_meas2[_df_meas2['_dtx'] <  300].head(10),
    _df_meas2[_df_meas2['_dtx'].between( 300,  700)].head(10),
    _df_meas2[_df_meas2['_dtx'].between( 700, 1200)].head(10),
    _df_meas2[_df_meas2['_dtx'].between(1200, 2000)].head(10),
    _df_meas2[_df_meas2['_dtx'] > 2000].head(10),
]
_df_test = pd.concat(_bands).drop_duplicates(subset='name').reset_index(drop=True)
print(f"  Testing {len(_df_test)} receivers across 5 distance bands:")
print(f"  Full physics: LOS + reflection + diffraction + edge_diffraction + scattering")
print(f"  Samples per RX: {NUM_SAMPLES_PS//1_000_000}M  |  max_depth: {MAX_DEPTH}  |  scat_keep_prob: 0.001")

_results50 = []
for _, _mrow in _df_test.iterrows():
    _lx = float(_mrow['_lx']); _ly = float(_mrow['_ly']); _d = float(_mrow['_dtx'])
    _rssi_meas = float(_mrow['local_measurement_dBm'])
    _pl_meas   = float(_mrow['path_loss_dB'])
    _rx_name   = str(_mrow['name'])
    # Terrain-correct height — priority: scene.receivers > DTM tiles > DEM fallback
    # scene.receivers has authoritative terrain-corrected z from the RX placement cell.
    _gz4 = None
    if _rx_name in scene.receivers:
        _gz4 = _safe(scene.receivers[_rx_name].position[2]) - RX_AGL_M
    if _gz4 is None and os.path.exists(RX_CSV):
        try:
            _df_rxcsv = pd.read_csv(RX_CSV)
            _row_match = _df_rxcsv[_df_rxcsv['name'] == _rx_name]
            if len(_row_match) > 0 and 'z_m' in _df_rxcsv.columns:
                _gz4 = float(_row_match.iloc[0]['z_m']) - RX_AGL_M
        except Exception: pass
    if _gz4 is None and "_dtm_at_wgs" in dir():
        _lon4, _lat4 = float(_mrow["lon"]), float(_mrow["lat"])
        _gz4 = _dtm_at_wgs(_lon4, _lat4) - _SCENE_CENTRE_DTM
    if _gz4 is None:
        # Last resort: DEM elevation — avoids ray_cast_ground_z which hits building roofs
        try:
            _ux4f, _uy4f = gps_to_utm.transform(float(_mrow["lon"]), float(_mrow["lat"]))
            _lx4f = _ux4f - utm_center_x; _ly4f = _uy4f - utm_center_y
            _gz4 = get_dem_elevation(_lx4f, _ly4f) - _SCENE_ORIGIN_ELEV  # ASL → scene-local
        except Exception:
            _gz4 = 0.0
    _rx_z_final = _gz4 + RX_AGL_M
    print(f'    [{_rx_name}] ground_z={_gz4:.2f}m  rx_z={_rx_z_final:.2f}m')
    _rx = Receiver(name=_rx_name, position=[_lx, _ly, _rx_z_final])

    # Remove all, add just this one, compute paths
    for _n in list(scene.receivers.keys()): scene.remove(_n)
    scene.add(_rx)
    try:
        _t0 = time.time()
        _paths = scene.compute_paths(
            max_depth        = MAX_DEPTH,
            los              = True,
            reflection       = True,
            diffraction      = True,
            edge_diffraction = True,
            scattering       = True,
            scat_keep_prob   = 0.001,
            num_samples      = NUM_SAMPLES_PS
        )
        _dt = time.time() - _t0
        _a_raw = _paths.a
        if isinstance(_a_raw, tuple): _a = _a_raw[0].numpy() + 1j*_a_raw[1].numpy()
        else: _a = np.array(_a_raw)
        # Retry once with 2x samples on 0-path short-range NLOS (cap 20M to avoid OOM)
        if float(np.sum(np.abs(_a)**2)) == 0 and _d < 600:
            try:
                _n_retry = min(NUM_SAMPLES_PS * 2, 20_000_000)
                _paths2 = scene.compute_paths(
                    max_depth=MAX_DEPTH, los=True, reflection=True,
                    diffraction=True, edge_diffraction=True,
                    scattering=True, scat_keep_prob=0.01,
                    num_samples=_n_retry
                )
                _a2_raw = _paths2.a
                if isinstance(_a2_raw, tuple): _a2 = _a2_raw[0].numpy() + 1j*_a2_raw[1].numpy()
                else: _a2 = np.array(_a2_raw)
                if float(np.sum(np.abs(_a2)**2)) > 0:
                    _a = _a2
                    _paths = _paths2
                    print(f"    [retry 2x({_n_retry//1_000_000}M) → {int(np.sum(np.abs(_a)>0))} paths]")
            except Exception as _retry_e:
                print(f"    [retry failed: {_retry_e}]")
        # Scale scatter path amplitudes by sqrt(1/scat_keep_prob) to correct
        # for the Monte-Carlo subsampling applied during path tracing.
        _scat_kp = 0.001  # must match scat_keep_prob above
        try:
            _types = _paths.types
            if isinstance(_types, (list, tuple)): _types = np.array(_types)
            else: _types = np.array(_types)
            _scat_mask = (_types == 3)  # EM_SCATTERING = 3 in Sionna 0.19
            if _scat_mask.any():
                _a[_scat_mask] *= np.sqrt(1.0 / _scat_kp)
        except Exception:
            pass  # paths.types unavailable — skip correction
        _pwr = float(np.sum(np.abs(_a)**2))
        if _pwr > 0:
            _pl_sim  = -10*math.log10(_pwr)
            # Correct Sionna formula: paths.a includes TX+RX antenna gains
            # → use TX_CONDUCTED (not EIRP) and RX_EXTRA (not SYS_GAIN)
            _rssi_sim = TX_CONDUCTED_DBM - _pl_sim + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB
            _n_paths = int(np.sum(np.abs(_a) > 0))
            # Filter numerical noise: single near-zero path gives -200 dBm garbage
            if _rssi_sim < -150.0:
                _pl_sim = _rssi_sim = float('nan')
                _n_paths = 0
        else:
            _pl_sim = _rssi_sim = float('nan')
            _n_paths = 0
        _fspl = 20*math.log10(4*math.pi*_d*FREQUENCY_HZ/C) if _d > 0 else 0
        _results50.append({
            'name': _rx.name, 'dist_m': _d,
            'rssi_sim': _rssi_sim, 'rssi_meas': _rssi_meas,
            'pl_sim': _pl_sim, 'pl_meas': _pl_meas, 'fspl': _fspl,
            'n_paths': _n_paths, 'dt': _dt
        })
        print(f"  {_rx.name:<12} d={_d:5.0f}m  paths={_n_paths:3d}  "
              f"RSSI sim={_rssi_sim:6.1f} meas={_rssi_meas:6.1f}  "
              f"err={_rssi_sim-_rssi_meas:+5.1f}  PL-FSPL={_pl_sim-_fspl:+5.1f} dB  ({_dt:.1f}s)")
    except Exception as _e:
        print(f"  {_rx.name:<12} ERROR: {_e}")

# Re-add all receivers
for _n in list(scene.receivers.keys()): scene.remove(_n)
for _rx in _rxlist: scene.add(_rx)

# Summary
_r10_raw = pd.DataFrame(_results50).dropna(subset=['rssi_sim','rssi_meas'])

# ── Clamp: remove physically impossible or near-field anomalies ───────────
# Exclude: sim RSSI > -5 dBm (near-field / impossible) or dist < 50m
_RSSI_MAX_VALID  = -5.0   # dBm — above this is near-field artefact
_DIST_MIN_VALID  = 50.0   # m   — below this is near-field
_r10_clamped = _r10_raw[
    (_r10_raw['rssi_sim'] <= _RSSI_MAX_VALID) &
    (_r10_raw['dist_m']   >= _DIST_MIN_VALID)
].copy()

_n_removed = len(_r10_raw) - len(_r10_clamped)
if _n_removed:
    print(f"  Clamped {_n_removed} anomalous RX (sim RSSI>{_RSSI_MAX_VALID}dBm or dist<{_DIST_MIN_VALID}m):")
    for _, _rr in _r10_raw[~_r10_raw.index.isin(_r10_clamped.index)].iterrows():
        print(f"    {_rr['name']:<14} d={_rr['dist_m']:4.0f}m  sim={_rr['rssi_sim']:+.1f}dBm  meas={_rr['rssi_meas']:+.1f}dBm  → EXCLUDED")

_r50 = _r10_clamped
if len(_r50):
    _bias = (_r50['rssi_sim'] - _r50['rssi_meas']).mean()
    _rmse = math.sqrt(((_r50['rssi_sim'] - _r50['rssi_meas'])**2).mean())
    print(f"\n  Valid RX summary ({len(_r50)} receivers, dist≥{_DIST_MIN_VALID:.0f}m):")
    print(f"  bias={_bias:+.1f} dB  RMSE={_rmse:.1f} dB")
    print(f"  PL vs FSPL   :  mean={(_r50['pl_sim']-_r50['fspl']).mean():+.1f} dB  "
          f"(urban overhead, expect +5 to +20 dB)")
    # Distance-band breakdown
    print(f"\n  Distance-band breakdown:")
    print(f"  {'Band':<12} {'N':>4} {'Bias(dB)':>10} {'RMSE(dB)':>10} {'Mean paths':>11}")
    print(f"  {'-'*52}")
    for _bname, _bmin, _bmax in [('<300m',0,300),('300-700m',300,700),
                                   ('700-1200m',700,1200),('1.2-2km',1200,2000),('>2km',2000,9999)]:
        _rb = _r50[(_r50['dist_m']>=_bmin) & (_r50['dist_m']<_bmax)]
        if len(_rb) == 0: continue
        _berr = _rb['rssi_sim'] - _rb['rssi_meas']
        _bbias = _berr.mean(); _brmse = float(np.sqrt((_berr**2).mean()))
        _bpaths = _rb['n_paths'].mean()
        print(f"  {_bname:<12} {len(_rb):>4} {_bbias:>+10.1f} {_brmse:>10.1f} {_bpaths:>11.0f}")
    if abs(_bias) > 10:
        print(f"\n  ⚠ Large bias — material calibration needed (diff-RT)")
    elif _rmse > 10:
        print(f"  ⚠ Large scatter — likely geometry/height mismDone — check STEP 1-4 results above before running full CELL 9b")

# ── STEP 5: Sim vs Ofcom — matched by RX ID ──────────────────────────────────
print("\n" + "─" * 60)
print("STEP 5 — Sim vs Ofcom RSSI (matched by RX ID)")
print("─" * 60)

try:
    # Use _results50 from STEP 4 — already matched sim+meas by RX ID
    _df5 = pd.DataFrame([r for r in _results50
                         if not math.isnan(r.get('rssi_sim', float('nan')))
                         and not math.isnan(r.get('rssi_meas', float('nan')))])
    if len(_df5) == 0:
        print("  No valid sim results — run STEP 4 first.")
    else:
        print(f"  Receivers with sim+meas: {len(_df5)}  (excluded NaN sim: {len(_results50)-len(_df5)})")
        print()
        _err5 = _df5['rssi_sim'] - _df5['rssi_meas']
        _bias5 = _err5.mean()
        _rmse5 = math.sqrt((_err5**2).mean())
        print(f"  Overall  bias={_bias5:+.1f} dB   RMSE={_rmse5:.1f} dB   N={len(_df5)}")
        print()

        # Per-distance-band breakdown
        _bands = [(0,100,'<100m'), (100,500,'100–500m'), (500,1200,'500m–1.2km'),
                  (1200,2500,'>1.2km')]
        print(f"  {'Band':<12} {'N':>4}  {'Sim RSSI':>10}  {'Meas RSSI':>10}  {'Bias':>8}  {'RMSE':>8}  {'Paths':>7}")
        print(f"  {'-'*12} {'-'*4}  {'-'*10}  {'-'*10}  {'-'*8}  {'-'*8}  {'-'*7}")
        for d0, d1, lbl in _bands:
            _s = _df5[(_df5['dist_m'] >= d0) & (_df5['dist_m'] < d1)]
            if len(_s) == 0:
                continue
            _e = _s['rssi_sim'] - _s['rssi_meas']
            print(f"  {lbl:<12} {len(_s):>4}  {_s['rssi_sim'].mean():>+9.1f}  "
                  f"{_s['rssi_meas'].mean():>+9.1f}  {_e.mean():>+7.1f}  "
                  f"{math.sqrt((_e**2).mean()):>7.1f}  {_s['n_paths'].mean():>7.1f}")
        print()

        # Per-RX detail table
        print(f"  {'RX ID':<14} {'Dist':>7}  {'Sim':>8}  {'Meas':>8}  {'Err':>7}  {'Paths':>6}")
        print(f"  {'-'*14} {'-'*7}  {'-'*8}  {'-'*8}  {'-'*7}  {'-'*6}")
        for _, _r in _df5.sort_values('dist_m').iterrows():
            print(f"  {_r['name']:<14} {_r['dist_m']:>6.0f}m  "
                  f"{_r['rssi_sim']:>+7.1f}  {_r['rssi_meas']:>+7.1f}  "
                  f"{_r['rssi_sim']-_r['rssi_meas']:>+6.1f}  {int(_r['n_paths']):>6}")

except Exception as _e5:
    print(f"  STEP 5 error: {_e5}")
    import traceback; traceback.print_exc()


## Cell 8 — Path Solver (900 MHz, Batched)

Full path solver with per-ray extraction, adaptive samples, and CSV output.
Same logic as CELL 9b of the main notebook.

In [ ]:
# ====================================================================
# CELL 8 — PATH SOLVER WITH PER-RAY EXTRACTION  [900 MHz / Sionna 0.19]
# ====================================================================
import gc, time, os, math
import numpy as np, pandas as pd
import drjit as dr
from datetime import datetime

print('=' * 70)
print('CELL 8 — PATH SOLVER  [900 MHz / Sionna 0.19]')
print('=' * 70)

_safe = lambda v: float(v) if not hasattr(v, 'numpy') else float(v.numpy())

# ── Configuration ─────────────────────────────────────────────────────────────
SAVE_PER_RAY    = True
MAX_RAYS_PER_RX = 300
MAX_SAMPLES_PS  = 20_000_000   # hard cap for OOM safety

_is_cpu_ps = 'llvm' in mi.variant() or 'scalar' in mi.variant()
PS_CONFIG_BASE = dict(
    max_depth        = MAX_DEPTH,
    los              = True,
    reflection       = True,
    diffraction      = not _is_cpu_ps,
    edge_diffraction = True,
    scattering       = True,
    scat_keep_prob   = SCAT_KEEP_PROB,
)

tx     = list(scene.transmitters.values())[0]
tx_pos = np.array([_safe(tx.position[0]), _safe(tx.position[1]), _safe(tx.position[2])])
C      = 3e8

print(f'  TX conducted    : {TX_CONDUCTED_DBM:.1f} dBm  |  RX extra: {RX_EXTRA_GAIN_DB:.1f} dB  |  Site corr: {SITE_CORRECTION_DB:.1f} dB')
print(f'  TX position     : ({tx_pos[0]:.1f}, {tx_pos[1]:.1f}, {tx_pos[2]:.1f}) m')
print(f'  Batch size      : {BATCH_SIZE} receivers  |  Base samples: {NUM_SAMPLES_PS:,}')
for k, v in PS_CONFIG_BASE.items():
    print(f'  {k:15s}: {v}')

def adaptive_samples(dist_m):
    if dist_m > 9000:   return min(NUM_SAMPLES_PS * 2, MAX_SAMPLES_PS)
    elif dist_m > 5000: return min(int(NUM_SAMPLES_PS * 1.5), MAX_SAMPLES_PS)
    else:               return NUM_SAMPLES_PS

def extract_amplitudes(paths):
    a = paths.a
    if isinstance(a, tuple): a_np = a[0].numpy() + 1j * a[1].numpy()
    else:                    a_np = np.array(a)
    a_np = np.squeeze(a_np)
    while a_np.ndim > 2: a_np = a_np[..., 0]
    if a_np.ndim == 1: a_np = a_np[np.newaxis, :]
    return a_np

def extract_tau(paths, num_rx, num_paths):
    tau = getattr(paths, 'tau', None)
    if tau is None: return np.full((num_rx, num_paths), np.nan, np.float32)
    try:
        t = tau.numpy() if hasattr(tau, 'numpy') else np.array(tau)
        t = np.squeeze(t)
        while t.ndim > 2: t = t[..., 0]
        if t.ndim == 1: t = t[np.newaxis, :]
        return t
    except Exception: return np.full((num_rx, num_paths), np.nan, np.float32)

def summary_metrics(a_row):
    pwr = np.abs(a_row) ** 2; valid = pwr > 1e-30
    if not np.any(valid): return np.nan, np.nan, np.nan, 0
    pv = pwr[valid]; av = a_row[valid]
    best_pl  = -10 * np.log10(np.max(pv))
    incoh_pl = -10 * np.log10(np.sum(pv))
    coh_pwr  = np.abs(np.sum(av)) ** 2
    coh_pl   = -10 * np.log10(coh_pwr) if coh_pwr > 1e-30 else np.nan
    return best_pl, incoh_pl, coh_pl, int(np.sum(valid))

def ray_type_heuristic(path_len, los_dist, pwr, max_pwr):
    if los_dist > 0 and abs(path_len - los_dist) / los_dist < 0.01: return 'LOS'
    ratio = pwr / max_pwr if max_pwr > 0 else 0; excess = path_len - los_dist
    if excess < 50  and ratio > 0.01:  return 'REFLECTION'
    if excess >= 50 and ratio > 0.001: return 'MULTI_REFLECTION'
    if ratio < 0.01:                   return 'DIFFRACTION'
    if ratio < 0.001:                  return 'SCATTERING'
    return 'UNKNOWN'

def run_batch(batch, cfg):
    for nm in list(scene.receivers.keys()): scene.remove(nm)
    for rx in batch: scene.add(rx)
    try:
        paths = scene.compute_paths(**cfg)
    except Exception as _oom:
        if any(k in str(_oom).lower() for k in ['oom', 'resource exhausted', 'memory']):
            paths = scene.compute_paths(**{**cfg, 'num_samples': 500_000})
        else:
            raise
    return paths

# ── Sort by distance ──────────────────────────────────────────────────────────
_all_rx = list(receivers)
total   = len(_all_rx)
tx_pos2d = tx_pos[:2]
_all_rx.sort(key=lambda rx: float(np.linalg.norm(
    [_safe(rx.position[0]) - tx_pos2d[0], _safe(rx.position[1]) - tx_pos2d[1]])))

ts          = datetime.now().strftime('%Y%m%d_%H%M%S')
summary_csv = os.path.join(OUT_DIR, f'path_solver_summary_900_{ts}.csv')
per_ray_csv = os.path.join(OUT_DIR, f'path_solver_per_ray_900_{ts}.csv') if SAVE_PER_RAY else None

summary_rows = []; per_ray_rows = []; errors = 0
t0 = time.time()

print(f'\nProcessing {total} receivers in batches of {BATCH_SIZE} ...')

for b_start in range(0, total, BATCH_SIZE):
    batch    = _all_rx[b_start : b_start + BATCH_SIZE]
    max_dist = max(float(np.linalg.norm(
        [_safe(rx.position[0]) - tx_pos2d[0], _safe(rx.position[1]) - tx_pos2d[1]]))
        for rx in batch)
    n_samp = adaptive_samples(max_dist)
    cfg    = {**PS_CONFIG_BASE, 'num_samples': n_samp}

    paths = None; batch_paths = 0
    try:
        paths   = run_batch(batch, cfg)
        a_all   = extract_amplitudes(paths)
        batch_paths = int(np.sum(np.abs(a_all) ** 2 > 1e-30))
    except Exception as _e:
        print(f'  [WARN] Batch {b_start}: {_e}')
        paths = None; a_all = None; batch_paths = 0

    # Retry with 2× samples if 0 paths and short-range (<600 m)
    if paths is not None and batch_paths == 0 and max_dist < 600:
        try:
            paths2  = run_batch(batch, {**cfg, 'num_samples': min(n_samp * 2, MAX_SAMPLES_PS)})
            a_all2  = extract_amplitudes(paths2)
            bp2     = int(np.sum(np.abs(a_all2) ** 2 > 1e-30))
            if bp2 > 0:
                paths = paths2; a_all = a_all2; batch_paths = bp2
                print(f'  [retry 2x → {bp2} paths at {max_dist:.0f}m]')
        except Exception: pass

    if paths is None or batch_paths == 0:
        for rx in batch:
            los_d = float(np.linalg.norm(
                np.array([_safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])]) - tx_pos))
            summary_rows.append({'receiver': rx.name,
                'x_m': _safe(rx.position[0]), 'y_m': _safe(rx.position[1]),
                'z_m': _safe(rx.position[2]), 'dist_from_tx_m': los_d,
                'num_samples_used': n_samp, 'num_paths': 0,
                'path_loss_best_db': np.nan, 'path_loss_incoherent_db': np.nan,
                'path_loss_coherent_db': np.nan, 'rssi_best_dbm': np.nan,
                'rssi_incoherent_dbm': np.nan, 'rssi_coherent_dbm': np.nan})
        if paths is None: errors += len(batch)
        if paths is not None: del paths
        gc.collect()
        done = min(b_start + BATCH_SIZE, total)
        if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
            print(f'  [{done}/{total}]  {time.time()-t0:.0f}s  (0 paths — NLOS/far)')
        continue

    n_b, n_p  = a_all.shape
    tau_all   = extract_tau(paths, n_b, n_p)

    for i, rx in enumerate(batch):
        rx_pos  = np.array([_safe(rx.position[0]), _safe(rx.position[1]), _safe(rx.position[2])])
        los_d   = float(np.linalg.norm(rx_pos - tx_pos))
        idx     = i if i < n_b else n_b - 1
        best_pl, incoh_pl, coh_pl, n_valid = summary_metrics(a_all[idx])

        summary_rows.append({
            'receiver'                : rx.name,
            'x_m'                    : _safe(rx.position[0]),
            'y_m'                    : _safe(rx.position[1]),
            'z_m'                    : _safe(rx.position[2]),
            'dist_from_tx_m'         : los_d,
            'num_samples_used'       : n_samp,
            'num_paths'              : n_valid,
            'path_loss_best_db'      : best_pl,
            'path_loss_incoherent_db': incoh_pl,
            'path_loss_coherent_db'  : coh_pl,
            'rssi_best_dbm'          : min(TX_CONDUCTED_DBM - best_pl  + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB, -30.0) if not np.isnan(best_pl)  else np.nan,
            'rssi_incoherent_dbm'    : min(TX_CONDUCTED_DBM - incoh_pl + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB, -30.0) if not np.isnan(incoh_pl) else np.nan,
            'rssi_coherent_dbm'      : min(TX_CONDUCTED_DBM - coh_pl   + RX_EXTRA_GAIN_DB + SITE_CORRECTION_DB, -30.0) if not np.isnan(coh_pl)   else np.nan,
        })

        if SAVE_PER_RAY and n_valid > 0:
            a_row   = a_all[idx]; pwr_row = np.abs(a_row) ** 2
            order   = np.argsort(pwr_row)[::-1]; max_pwr = pwr_row[order[0]]
            strong_phase = np.angle(a_row[order[0]], deg=True)
            for rank, ray_i in enumerate(order[:MAX_RAYS_PER_RX]):
                ac = a_row[ray_i]; pwr_ray = float(pwr_row[ray_i])
                if pwr_ray <= 1e-30: break
                phase   = float(np.angle(ac, deg=True))
                ph_diff = (phase - strong_phase + 180) % 360 - 180
                delay   = float(tau_all[idx, ray_i]) if not np.isnan(tau_all[idx, ray_i]) else np.nan
                plen    = delay * C if not np.isnan(delay) else np.nan
                rtype   = ray_type_heuristic(plen, los_d, pwr_ray, max_pwr) if not np.isnan(plen or 0.0) else 'UNKNOWN'
                per_ray_rows.append({
                    'receiver': rx.name, 'rank': rank, 'ray_type': rtype,
                    'power_linear': pwr_ray, 'path_loss_db': -10 * np.log10(pwr_ray),
                    'amplitude_real': float(ac.real), 'amplitude_imag': float(ac.imag),
                    'phase_deg': phase, 'phase_diff_deg': ph_diff,
                    'constructive': 'STRONGEST' if rank == 0 else ('CONSTRUCTIVE' if abs(ph_diff) < 90 else 'DESTRUCTIVE'),
                    'delay_s': delay, 'path_length_m': plen,
                })

    del paths, a_all, tau_all
    gc.collect()
    done = min(b_start + BATCH_SIZE, total)
    if done % max(BATCH_SIZE, total // 10) < BATCH_SIZE or done == total:
        elapsed = time.time() - t0
        eta     = (total - done) / max(done / max(elapsed, 1e-9), 1e-9)
        print(f'  [{done}/{total}]  {elapsed:.0f}s elapsed  ETA {eta/60:.1f} min', flush=True)

# Restore all receivers
for nm in list(scene.receivers.keys()): scene.remove(nm)
for rx in _all_rx: scene.add(rx)

# ── Save ──────────────────────────────────────────────────────────────────────
df_ps = pd.DataFrame(summary_rows)
df_ps.to_csv(summary_csv, index=False)
print(f'\n  Summary  -> {summary_csv}')

if SAVE_PER_RAY and per_ray_rows:
    df_ray = pd.DataFrame(per_ray_rows)
    df_ray.to_csv(per_ray_csv, index=False)
    print(f'  Per-ray  -> {per_ray_csv}  ({len(df_ray):,} rays)')

elapsed = time.time() - t0
valid   = df_ps[df_ps['num_paths'] > 0]
nan_rx  = df_ps[df_ps['num_paths'] == 0]
print(f'\n  Total time      : {elapsed:.1f}s  |  Errors: {errors}')
print(f'  Receivers solved: {len(valid)}/{total} ({100*len(valid)/max(total,1):.1f}%)')
print(f'  Zero-path (NaN) : {len(nan_rx)}')

for col, lbl in [('path_loss_incoherent_db','PL incoherent'),('rssi_incoherent_dbm','RSSI incoher.')]:
    v = df_ps[col].dropna()
    if len(v):
        print(f'  {lbl:20s}: mean={v.mean():.1f}  std={v.std():.1f}  min={v.min():.1f}  max={v.max():.1f}')

import matplotlib.pyplot as plt
_df = df_ps.copy(); _df['dist_km'] = _df['dist_from_tx_m'] / 1000
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
_v = _df.dropna(subset=['path_loss_incoherent_db'])
axes[0].scatter(_v['dist_km'], _v['path_loss_incoherent_db'], s=5, alpha=0.5, c='steelblue')
axes[0].set(xlabel='Distance (km)', ylabel='Path Loss (dB)', title='Incoherent PL vs Distance'); axes[0].grid(alpha=0.3)
axes[1].hist(_df['rssi_incoherent_dbm'].dropna(), bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].set(xlabel='RSSI (dBm)', ylabel='Count', title='RSSI Distribution'); axes[1].grid(alpha=0.3)
axes[2].scatter(_df['dist_km'], _df['num_paths'], s=5, alpha=0.4, c='seagreen')
axes[2].set(xlabel='Distance (km)', ylabel='Num paths', title='Paths vs Distance'); axes[2].grid(alpha=0.3)
axes[2].set_yscale('symlog')
plt.suptitle(f'Path Solver 900 MHz — {len(df_ps)} receivers', fontsize=12)
plt.tight_layout()
_p = os.path.join(OUT_DIR, 'cell8_path_solver_900mhz.png')
plt.savefig(_p, dpi=150, bbox_inches='tight'); plt.show()
print(f'Plot saved → {_p}')


## Cell 9 — Compare vs Measurements

In [ ]:
if not MEASUREMENT_CSV or not os.path.exists(MEASUREMENT_CSV):
    print('Set MEASUREMENT_CSV in Cell 1 to compare vs measurements.')
else:
    df_meas = pd.read_csv(MEASUREMENT_CSV)
    df_merge = df_sim.merge(df_meas[['name','local_measurement_dBm']], on='name', how='inner')
    df_merge = df_merge.dropna(subset=['rssi_sim_dbm','local_measurement_dBm'])
    df_merge['err'] = df_merge['rssi_sim_dbm'] - df_merge['local_measurement_dBm']

    bias = df_merge['err'].mean()
    rmse = (df_merge['err']**2).mean()**0.5
    print(f'Receivers compared : {len(df_merge)}')
    print(f'Bias (sim-meas)    : {bias:+.2f} dB')
    print(f'RMSE               : {rmse:.2f} dB')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].scatter(df_merge['dist_m']/1000, df_merge['err'], s=8, alpha=0.5)
    axes[0].axhline(0, color='red', lw=1)
    axes[0].set_xlabel('Distance (km)'); axes[0].set_ylabel('Error (dB)')
    axes[0].set_title(f'Prediction error vs distance  (bias={bias:+.1f} dB, RMSE={rmse:.1f} dB)')

    axes[1].scatter(df_merge['local_measurement_dBm'], df_merge['rssi_sim_dbm'], s=8, alpha=0.5)
    _lo = min(df_merge['local_measurement_dBm'].min(), df_merge['rssi_sim_dbm'].min())
    _hi = max(df_merge['local_measurement_dBm'].max(), df_merge['rssi_sim_dbm'].max())
    axes[1].plot([_lo,_hi],[_lo,_hi],'r--',lw=1)
    axes[1].set_xlabel('Measured RSSI (dBm)'); axes[1].set_ylabel('Simulated RSSI (dBm)')
    axes[1].set_title('Sim vs Measured  (900 MHz)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'rssi_compare_900mhz.png'), dpi=120)
    plt.show()